# MNIST classification using the OpenEye accelerator (Pytorch version)

This notebook demonstrates how to perform MNIST digit classification 
using a PyTorch model optimized for the OpenEye accelerator. 
It covers loading the dataset, defining the model architecture, 
training the model, deploying it on the OpenEye accelerator, and 
evaluating its performance.



In [4]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import matplotlib.pyplot as plt
import numpy as np

print(f"PyTorch Version: {torch.__version__}")


PyTorch Version: 2.9.0


In [ ]:
""" Define the neural network architecture.

In this tutorial, we use a host device for training and the OpenEye accelerator
purely for inference. Hence, we define a simple feedforward neural network and 
train it on the host device with the available hardware acceleration. 

PyTorch can run on CPU, GPU or MPS. Here, we check for GPU or MPS availability.
"""
device = torch.device("cuda" if torch.cuda.is_available() 
                      else "mps" if torch.backends.mps.is_available() 
                      else "cpu")

print(f"Using device: {device}")

In [ ]:
# Hyperparameters are settings that influence training
BATCH_SIZE = 64          # Number of images per training step
LEARNING_RATE = 0.001    # Step size when learning (too large = unstable, too small = slow)
EPOCHS = 5               # How many times to iterate through entire dataset
INPUT_SIZE = 28 * 28     # MNIST images are 28x28 pixels = 784 pixels
HIDDEN_SIZE = 128        # Number of neurons in hidden layer
NUM_CLASSES = 10         # Digits 0-9 = 10 classes

In [ ]:
# Transform: Converts images to PyTorch tensors and normalizes them
# Normalization: (value - mean) / standard deviation
# This helps the neural network learn better
transform = transforms.Compose([
    transforms.ToTensor(),                    # Image to tensor (0-255 -> 0-1)
    transforms.Normalize((0.1307,), (0.3081,))  # Normalization with MNIST statistics
])

In [ ]:
# Download datasets (first time only) and load
train_dataset = datasets.MNIST(
    root='./data',           # Storage location
    train=True,              # Training data
    download=True,           # Download if not present
    transform=transform      # Apply transformations
)

test_dataset = datasets.MNIST(
    root='./data',
    train=False,             # Test data
    download=True,
    transform=transform
)

In [ ]:
# DataLoader: Loads data in batches and shuffles them
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,            # Shuffle data for each epoch
    num_workers=2            # Parallel workers for loading
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

In [ ]:
print(f"Training data: {len(train_dataset)} images")
print(f"Test data: {len(test_dataset)} images")
print(f"Number of batches per epoch: {len(train_loader)}")

In [ ]:
from mnist_conv_net import SimpleMNISTConvNet



In [ ]:
"""Create model and move to device"""
model = SimpleMNISTConvNet().to(device)  
print("\nModel Architecture:")
print(model)

# Count number of parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nTotal parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")


In [ ]:
# Loss function: Measures how wrong the predictions are
# CrossEntropyLoss combines LogSoftmax and NLLLoss
# Perfect for classification problems!
criterion = nn.CrossEntropyLoss()

# Optimizer: Updates the network's weights
# Adam is an improved version of Stochastic Gradient Descent
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

# Optional: Learning Rate Scheduler (reduces learning rate over time)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=3, gamma=0.7)

In [ ]:
def train(model, device, train_loader, criterion, optimizer, epoch):
    """
    Trains the model for one epoch
    
    Args:
        model: The neural network
        device: CPU or CUDA
        train_loader: DataLoader with training data
        criterion: Loss function
        optimizer: Optimizer
        epoch: Current epoch number
    
    Returns:
        Average loss
    """
    model.train()  # Sets model to training mode (important for Dropout, BatchNorm)
    
    running_loss = 0.0
    correct = 0
    total = 0
    
    for batch_idx, (data, target) in enumerate(train_loader):
        # Move data to device (CPU or GPU)
        data, target = data.to(device), target.to(device)
        
        # 1. Zero the gradients (important!)
        optimizer.zero_grad()
        
        # 2. Forward Pass: Compute predictions
        output = model(data)
        
        # 3. Calculate loss
        loss = criterion(output, target)
        
        # 4. Backward Pass: Compute gradients
        loss.backward()
        
        # 5. Optimizer Step: Update weights
        optimizer.step()
        
        # Statistics
        running_loss += loss.item()
        _, predicted = torch.max(output.data, 1)
        total += target.size(0)
        correct += (predicted == target).sum().item()
        
        # Show progress every 100 batches
        if batch_idx % 100 == 0:
            print(f'Epoch: {epoch} [{batch_idx * len(data)}/{len(train_loader.dataset)} '
                  f'({100. * batch_idx / len(train_loader):.0f}%)]\t'
                  f'Loss: {loss.item():.6f}')
    
    avg_loss = running_loss / len(train_loader)
    accuracy = 100. * correct / total
    
    print(f'\nTraining: Average Loss: {avg_loss:.4f}, '
          f'Accuracy: {correct}/{total} ({accuracy:.2f}%)\n')
    
    return avg_loss

In [ ]:
def test(model, device, test_loader, criterion):
    """
    Evaluates the model on test data
    
    Args:
        model: The neural network
        device: CPU or CUDA
        test_loader: DataLoader with test data
        criterion: Loss function
    
    Returns:
        Average loss and accuracy
    """
    model.eval()  # Sets model to evaluation mode (disables Dropout)
    
    test_loss = 0
    correct = 0
    
    # torch.no_grad() disables gradient computation (saves memory and time)
    with torch.no_grad():
        for data, target in test_loader:
            data, target = data.to(device), target.to(device)
            
            # Forward Pass
            output = model(data)
            
            # Sum up loss
            test_loss += criterion(output, target).item()
            
            # Prediction is the class with highest probability
            pred = output.argmax(dim=1, keepdim=True)
            correct += pred.eq(target.view_as(pred)).sum().item()
    
    test_loss /= len(test_loader)
    accuracy = 100. * correct / len(test_loader.dataset)
    
    print(f'Test: Average Loss: {test_loss:.4f}, '
          f'Accuracy: {correct}/{len(test_loader.dataset)} ({accuracy:.2f}%)\n')
    
    return test_loss, accuracy

In [ ]:
print("\n" + "="*70)
print("STARTING TRAINING")
print("="*70 + "\n")

train_losses = []
test_losses = []
test_accuracies = []

for epoch in range(1, EPOCHS + 1):
    print(f"\n{'='*70}")
    print(f"EPOCH {epoch}/{EPOCHS}")
    print(f"{'='*70}")
    
    # Train
    train_loss = train(model, device, train_loader, criterion, optimizer, epoch)
    train_losses.append(train_loss)
    
    # Test
    test_loss, test_acc = test(model, device, test_loader, criterion)
    test_losses.append(test_loss)
    test_accuracies.append(test_acc)
    
    # Adjust learning rate
    scheduler.step()
    print(f"Learning Rate: {scheduler.get_last_lr()[0]:.6f}")

print("\n" + "="*70)
print("TRAINING COMPLETED!")
print("="*70)
print(f"Best Test Accuracy: {max(test_accuracies):.2f}%")

In [ ]:

def visualize_predictions(model, device, test_loader, num_images=10):
    """Shows examples with predictions"""
    model.eval()
    
    # Get one batch
    data_iter = iter(test_loader)
    images, labels = next(data_iter)
    
    images = images.to(device)
    with torch.no_grad():
        outputs = model(images)
        _, predictions = torch.max(outputs, 1)
    
    # Back to CPU for Matplotlib
    images = images.cpu()
    predictions = predictions.cpu()
    
    # Plot
    fig, axes = plt.subplots(2, 5, figsize=(15, 6))
    axes = axes.ravel()
    
    for idx in range(num_images):
        img = images[idx].squeeze()
        true_label = labels[idx].item()
        pred_label = predictions[idx].item()
        
        axes[idx].imshow(img, cmap='gray')
        color = 'green' if true_label == pred_label else 'red'
        axes[idx].set_title(f'True: {true_label}\nPred: {pred_label}', 
                           color=color, fontweight='bold', fontsize=11)
        axes[idx].axis('off')
    
    plt.tight_layout()
    print("Prediction examples saved: predictions.png")


# Create visualizations
visualize_predictions(model, device, test_loader)

In [ ]:
# Save model completely, so that it can be reloaded later without knowing architecture
model_path = 'mnist_unquantized_model.pth'
torch.save({
    'epoch': EPOCHS,
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'test_accuracy': max(test_accuracies),
}, model_path)
print(f"\nModel saved: {model_path}")

In [ ]:
def predict_single_image(model, image_tensor, device):
    """
    Makes a prediction for a single image
    
    Args:
        model: Trained model
        image_tensor: Image as tensor (1, 28, 28)
        device: CPU or CUDA
    
    Returns:
        Predicted class and probabilities
    """
    model.eval()
    
    # Add batch dimension: (1, 28, 28) -> (1, 1, 28, 28)
    if image_tensor.dim() == 3:
        image_tensor = image_tensor.unsqueeze(0)
    
    image_tensor = image_tensor.to(device)
    
    with torch.no_grad():
        output = model(image_tensor)
        probabilities = F.softmax(output, dim=1)
        predicted_class = output.argmax(dim=1).item()
        confidence = probabilities[0][predicted_class].item()
    
    return predicted_class, confidence, probabilities.cpu().numpy()[0]


# Example: Prediction for first test image
test_image, test_label = test_dataset[0]
predicted, confidence, probs = predict_single_image(model, test_image, device)

In [ ]:
import torchao
print(torchao.__version__)  # Print torchao version

In [ ]:
"""Before we can map the model to the OpenEye accelerator, we need to quantize
it to 8-bit integer arithmetic.

Hence, we define a function to quantize the trained model using PyTorch's static
quantization. The quantization performed here will convert the floating-point
weights and activations of the model to 8-bit integers in some simple "vanilla"
manner. Other more advanced quantization techniques may yield better accuracy,
but are beyond the scope of this demo. 
"""

def quantize_model(model, test_loader, device):
    """
    Quantizes a PyTorch model to 8-bit integer arithmetic
    Args:
        model: Trained PyTorch model
        test_loader: DataLoader for calibration
        device: CPU or CUDA
    """
    import copy
    # take a few examples from the test set for calibration
    calibration_batches = 10
    calibration_data = []
    for i, (data, target) in enumerate(test_loader):
        if i >= calibration_batches:
            break
        calibration_data.append(data)

    # we need a tuple of tensors for calibration
    calibration_data = tuple(calibration_data)

    non_quant_model = copy.deepcopy(model)
    non_quant_model.to('cpu')  # Quantization typically runs on CPU
    non_quant_model.eval()

    m = torch.export.export(non_quant_model, calibration_data).module()

    from torchao.quantization.pt2e.quantize_pt2e import (
        prepare_pt2e,
        convert_pt2e)

    from executorch.backends.xnnpack.quantizer.xnnpack_quantizer import (
        get_symmetric_quantization_config,
        XNNPACKQuantizer)
    
    quantizer = XNNPACKQuantizer().set_global(get_symmetric_quantization_config())
    m = prepare_pt2e(m, quantizer)

    m = convert_pt2e(m)

    return quantized_model

quantized_model = quantize_model(model, test_loader, device)

quantized_model.conv1

quantized_model.fc1


In [ ]:
"""Just for info, we define a function to compare the file sizes of the original
float32 model vs the quantized int8 model."""

def compare_model_sizes(original_model, quantized_model):
    """
    Compare file sizes of original vs quantized model
    
    Args:
        original_model: Original float32 model
        quantized_model: Quantized int8 model
    """
    import os
    
    # Save both models temporarily
    torch.save(original_model.state_dict(), 'original_model.pth')
    torch.save(quantized_model.state_dict(), 'quantized_model.pth')
    
    # Get file sizes
    original_size = os.path.getsize('original_model.pth')
    quantized_size = os.path.getsize('quantized_model.pth')
    
    print("\n" + "="*70)
    print("MODEL SIZE COMPARISON")
    print("="*70)
    print(f"Original model (float32):  {original_size / 1024:.2f} KB")
    print(f"Quantized model (int8):    {quantized_size / 1024:.2f} KB")
    print(f"Size reduction:            {(1 - quantized_size/original_size)*100:.1f}%")
    print(f"Compression ratio:         {original_size/quantized_size:.2f}x")
    
    # Cleanup
    os.remove('original_model.pth')
    os.remove('quantized_model.pth')


In [ ]:
def benchmark_inference_speed(original_model, quantized_model, test_loader, device):
    """
    Compare inference speed of original vs quantized model
    
    Args:
        original_model: Original float32 model
        quantized_model: Quantized int8 model
        test_loader: DataLoader with test data
        device: CPU or CUDA
    """
    import time
    
    print("\n" + "="*70)
    print("INFERENCE SPEED COMPARISON")
    print("="*70)
    
    # Get one batch for testing
    data_iter = iter(test_loader)
    test_data, _ = next(data_iter)
    
    # Benchmark original model
    original_model_cpu = original_model.cpu()
    original_model_cpu.eval()
    
    # Warmup
    with torch.no_grad():
        for _ in range(10):
            _ = original_model_cpu(test_data)
    
    # Measure
    num_iterations = 100
    start_time = time.time()
    with torch.no_grad():
        for _ in range(num_iterations):
            _ = original_model_cpu(test_data)
    original_time = (time.time() - start_time) / num_iterations * 1000  # ms
    
    # Benchmark quantized model
    quantized_model.eval()
    
    # Warmup
    with torch.no_grad():
        for _ in range(10):
            _ = quantized_model(test_data)
    
    # Measure
    start_time = time.time()
    with torch.no_grad():
        for _ in range(num_iterations):
            _ = quantized_model(test_data)
    quantized_time = (time.time() - start_time) / num_iterations * 1000  # ms
    
    print(f"Original model (float32):  {original_time:.3f} ms per batch")
    print(f"Quantized model (int8):    {quantized_time:.3f} ms per batch")
    print(f"Speedup:                   {original_time/quantized_time:.2f}x faster")
    print(f"Time saved:                {((original_time-quantized_time)/original_time)*100:.1f}%")


In [ ]:
def evaluate_quantized_model(quantized_model, test_loader):
    """
    Evaluate accuracy of quantized model
    
    Args:
        quantized_model: Quantized model to evaluate
        test_loader: DataLoader with test data
    
    Returns:
        Accuracy percentage
    """
    quantized_model.eval()
    correct = 0
    total = 0
    
    with torch.no_grad():
        for data, target in test_loader:
            output = quantized_model(data)
            pred = output.argmax(dim=1, keepdim=True)
            correct += pred.eq(target.view_as(pred)).sum().item()
            total += target.size(0)
    
    accuracy = 100. * correct / total
    return accuracy




In [ ]:
# Execute quantization
print("\nStarting quantization process...")
quantized_model = quantize_model(model, test_loader, device)

# Compare model sizes
compare_model_sizes(model, quantized_model)

# Benchmark inference speed
benchmark_inference_speed(model, quantized_model, test_loader, device)

# Evaluate quantized model accuracy
print("\n" + "="*70)
print("ACCURACY COMPARISON")
print("="*70)
print(f"Original model accuracy:   {max(test_accuracies):.2f}%")

quantized_accuracy = evaluate_quantized_model(quantized_model, test_loader)
print(f"Quantized model accuracy:  {quantized_accuracy:.2f}%")
print(f"Accuracy difference:       {abs(max(test_accuracies) - quantized_accuracy):.2f}%")

if quantized_accuracy >= max(test_accuracies) - 1.0:
    print("✅ Excellent! Accuracy loss is less than 1%")
elif quantized_accuracy >= max(test_accuracies) - 2.0:
    print("✅ Good! Accuracy loss is acceptable (1-2%)")
else:
    print("⚠️  Consider using Quantization Aware Training for better accuracy")


# Save quantized model
quantized_model_path = 'mnist_model_quantized.pth'
torch.save(quantized_model.state_dict(), quantized_model_path)
print(f"\nQuantized model saved: {quantized_model_path}")

